In [1]:
%pip install -q -U "fastmcp" gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 911.6/911.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.2/289.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.3/171.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.3/279.3 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:


%pip install -q -U fastmcp gradio


import asyncio
import gradio as gr
from fastmcp import FastMCP, Client


mcp = FastMCP("Business Tools Server")

# Tool 1: Calculator
@mcp.tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b

# Tool 2: Order lookup
@mcp.tool
def check_order(order_id: str) -> dict:
    """Retrieve the status and amount of a sample order."""
    orders = {
        "ORD101": {"status": "Shipped", "amount": 2500},
        "ORD102": {"status": "Processing", "amount": 1800},
        "ORD103": {"status": "Delivered", "amount": 3200}
    }

    return orders.get(
        order_id.upper(),
        {"error": "Order not found"}
    )


async def run_mcp_tool(tool_name, tool_input):
    async with Client(mcp) as client:
        result = await client.call_tool(tool_name, tool_input)
        return result

async def execute_tool(operation, value1, value2):
    try:
        if operation == "Add numbers":
            a = float(value1)
            b = float(value2)

            result = await run_mcp_tool(
                "add_numbers",
                {"a": a, "b": b}
            )

        elif operation == "Check order":
            order_id = str(value1).strip()

            if not order_id:
                return "Please enter an order ID."

            result = await run_mcp_tool(
                "check_order",
                {"order_id": order_id}
            )

        else:
            return "Select an operation."

        # Format the MCP result for display
        if hasattr(result, "content"):
            parts = []
            for item in result.content:
                if hasattr(item, "text"):
                    parts.append(item.text)
                else:
                    parts.append(str(item))
            return "\n".join(parts)

        return str(result)

    except ValueError:
        return "For addition, enter valid numbers."
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"

with gr.Blocks(title="MCP Business Tools") as demo:
    gr.Markdown("# MCP Business Tools Demo")
    gr.Markdown(
        "Use an MCP client to call tools exposed by the server."
    )

    operation = gr.Radio(
        choices=["Add numbers", "Check order"],
        value="Add numbers",
        label="Choose an operation"
    )

    value1 = gr.Textbox(
        label="First number / Order ID",
        placeholder="100 or ORD101"
    )

    value2 = gr.Textbox(
        label="Second number (addition only)",
        placeholder="250"
    )

    run_button = gr.Button("Call MCP Tool")
    output = gr.Textbox(label="Tool Result", lines=5)

    run_button.click(
        fn=execute_tool,
        inputs=[operation, value1, value2],
        outputs=output
    )


demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://920fc79df759546c6e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
